# S03: version, retrieve and retire knowledge

70-minute lab. Question: what changes when the policy changes, and how do you prevent retrieval from serving a convincing but unauthorised source?

In [ ]:
from pathlib import Path
import os, sys, json, tempfile
# Run from repository root, notebooks/, or a Colab clone.
candidates=[Path.cwd(), Path.cwd().parent, Path('/content/ai-platform-architect-labs')]
ROOT=next((p for p in candidates if (p/'engine').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open the extracted repository. Colab: follow README Setup first.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–15 min | Inspect source ownership.** These are fictional, short policy sections, not arbitrary paragraph fragments. Read the source Markdown files and manifest. The tenant, roles, approval and active flags are assigned by a trusted ingestion process. The model cannot supply these fields.

In [ ]:
from engine.evaluate import retrieval_suite, release_gate
ix=Index(); docs=read_data('policies.json'); ix.build('v1',docs)
for who in [BUYER,FINANCE,BEACON]:
    hits=ix.search('negotiated discount schedule',who)
    print(who.subject,[r['id'] for r in hits])
rows=retrieval_suite(ix)
print(json.dumps(rows,indent=2));print(release_gate(rows))

**15–30 min | Make a real data release.** Change the current standard payment term from 45 to 40 days in a new immutable version. The old version remains queryable for rollback. Verify that source identity, section and corpus digest travel with the index. A stale cache must not mask the version change.

In [ ]:
import copy
v2=copy.deepcopy(docs)
for d in v2:
    if d['id']=='A-TERMS': d['text']=d['text'].replace('45 days','40 days')
ix.build('v2',v2)
for version in ['v1','v2']:
    hit=ix.search('standard supplier payment terms',BUYER,version)[0]
    print(version,hit['id'],hit['text'])
print('Manifests:',ix.db.execute('SELECT version,model,dimensions,digest FROM indexes').fetchall())

**30–45 min | Break and repair a chunk.** Split the payment section halfway through the Nova exception. Retrieve both pieces and inspect whether a top-1 result loses the exception. Compare sentence/section boundaries and parent-section expansion. Record a design rule and its storage/token tradeoff; do not assume a universal chunk size.

In [ ]:
section=next(d for d in docs if d['id']=='A-TERMS')
text=section['text']; cut=text.index('Nova')+5
print('Artificial fragment 1:',text[:cut]);print('Artificial fragment 2:',text[cut:])
# Exercise: build an independent version with these two fragments and evaluate Nova's exception.
# Preserve ACL and provenance on both; use unique IDs and update evidence mappings deliberately.

**45–60 min | Real embeddings and persistence.** This optional execution branch is part of the planned live demonstration; the trainer must preflight it. Embeddings are produced by the configured provider and persisted in SQLite, then queried after reopening. No random/fabricated vector is used. Compare source recall and refusal behavior with lexical search. Positive cosine does not prove relevance; the vector path intentionally has no universal similarity cutoff.

In [ ]:
if LIVE and os.environ.get('EMBED_KEY'):
    embed=ModelClient.environment('EMBED')
    with tempfile.TemporaryDirectory() as td:
        path=Path(td)/'vectors.db'; vector_index=Index(path)
        vector_index.build('vec-v1',docs,embed);vector_index.close()
        vector_index=Index(path)
        measured=retrieval_suite(vector_index,'vec-v1',embed)
        print(json.dumps(measured,indent=2)); print(release_gate(measured))
        vector_index.close()
else:
    print('Real embedding comparison pending; lexical measurements above are actual search results.')
ix.close()

**60–70 min | Decide.** Produce a data lifecycle diagram covering ingest, verify, split, embed, publish, revoke, reindex and delete. Decide which queries need SQL, lexical search, semantic search or a hybrid. Explain why a production service needs approved embedding data handling, external identity, ingestion validation, scalable vector storage and revocation-aware cache invalidation. This small SQLite index teaches these boundaries; it does not demonstrate production scale.